### Notebook 13 — ABIDE multi-site comparison (site selection)

Purpose: justify choosing a SINGLE ABIDE site (NYU) for the near-Gaussian
real-data leg, by measuring per-site sample sizes and kurtosis and showing
that pooling multiple sites introduces scanner heterogeneity.

In [1]:
import sys, numpy as np
sys.path.insert(0, "..")                 # so "src" is importable (run from notebooks/)
from nilearn import datasets
from src.LCT import _kappa_hat, _zscore_columns

In [2]:
# A spread of ABIDE acquisition sites to compare before selecting one.
SITES = ["NYU", "UM_1", "USM", "UCLA_1", "LEUVEN_2", "PITT", "YALE", "TRINITY", "OHSU", "CALTECH"]
ATLAS = "rois_cc200"                      # Craddock-200, same as the final analysis

In [3]:
def subject_kappa(ts):
    ts = np.asarray(ts, dtype=float)
    if ts.ndim != 2 or ts.shape[0] < 10 or ts.shape[1] < 2:
        return None
    # drop dead (zero-variance) ROIs before measuring kurtosis
    keep = ts.std(axis=0) > 0
    if keep.sum() < 2:
        return None
    return _kappa_hat(_zscore_columns(ts[:, keep]))

In [4]:
print(f"{'site':10s} {'n_subj':>6s} {'ASD':>4s} {'CTL':>4s} {'kappa_hat':>10s} {'%>1.1':>6s}")
rows = []
for site in SITES:
    try:
        d = datasets.fetch_abide_pcp(
            SITE_ID=[site], pipeline="cpac",
            band_pass_filtering=True, global_signal_regression=False,
            derivatives=[ATLAS], quality_checked=True, verbose=0)
        ts_list = d[ATLAS]
        ks = [k for k in (subject_kappa(t) for t in ts_list) if k is not None]
        dx = np.asarray(d.phenotypic["DX_GROUP"])
        n_asd, n_ctl = int((dx == 1).sum()), int((dx == 2).sum())
        kmean = float(np.mean(ks)) if ks else float("nan")
        frac = float(np.mean(np.array(ks) > 1.1)) if ks else float("nan")
        print(f"{site:10s} {len(ts_list):6d} {n_asd:4d} {n_ctl:4d} {kmean:10.3f} {frac:6.2f}")
        rows.append((site, len(ts_list), kmean))
    except Exception as e:
        print(f"{site:10s} error: {e}")

site       n_subj  ASD  CTL  kappa_hat  %>1.1
NYU           172   74   98      1.014   0.05
UM_1           86   34   52      1.187   0.57
USM            67   43   24      1.067   0.13
UCLA_1         64   37   27      1.037   0.11
LEUVEN_2       28   12   16      1.056   0.21
PITT           50   24   26      1.030   0.10
YALE           41   22   19      1.040   0.10
TRINITY        44   19   25      1.018   0.11
OHSU           25   12   13      1.012   0.04
CALTECH        15    5   10      1.013   0.07
